# 물리적 리스크 PRF 세부 증가율 매핑 

`물리적리스크_할인할증.xlsx` 파일을 불러와서, `사업장별 PRF` 시트에 다음 항목을 추가하는 재현용 코드

- 사업장 주소 기반 유역/권역 근사 매핑
- `drr`, `rfr`, `cfr` 세부 위험별 증가율 부여
- 기존 PRF 구조를 유지한 10년/20년/30년 후 PRF 계산
- 기업별 면적가중 PRF 요약 시트 생성

In [1]:
# 필요한 패키지 불러오기
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 입력/출력 파일 경로
# 본인 환경에서는 input_path만 실제 파일 위치로 수정하면 됩니다.
input_path = Path("/Users/dapanman/Downloads/물리적리스크_할인할증 (1).xlsx")
output_path = Path("/Users/dapanman/Downloads/물리적리스크_PRF_세부증가율_매핑.xlsx")

input_path.exists(), input_path

(True, PosixPath('/Users/dapanman/Downloads/물리적리스크_할인할증 (1).xlsx'))

## 1. 원본 데이터 불러오기

`사업장별 PRF` 시트는 : 사업장별 주소, 면적, 위도/경도, `drr`, `rfr`, `cfr`, 기존 `PRF`가 들어 있는 핵심 시트


In [2]:
# 사업장별 PRF 시트를 DataFrame으로 불러오기
site_sheet = "사업장별 PRF"
df = pd.read_excel(input_path, sheet_name=site_sheet)

# 열 이름 확인
print(df.shape)
display(df.head())
print(df.columns.tolist())

(163, 15)


,기업명,사업장명,사업장 위치,면적(km²),위도,경도,rfr_score,cfr_score,drr_score,B 점수,drr 백분위,B 백분위,drr 팩터,B 팩터,PRF
0,삼성전자,수원사업장,경기도 수원시 영통구 삼성로 129(매탄동),1.30000,37.257643,127.053129,1.065214,2.072041,2.263827,2.072041,0.484568,0.472222,1.045370,1.041667,1.087037
1,삼성전자,서초사업장,서울특별시 서초구 서초대로74길 11(서초동),0.01340,37.496633,127.026914,0.562331,0.000000,1.965401,0.562331,0.030864,0.074074,0.909259,0.922222,0.831481
2,삼성전자,우면사업장,서울특별시 서초구 성촌길 33(우면동),0.03720,37.465025,127.022762,0.562331,0.000000,1.965401,0.562331,0.030864,0.074074,0.909259,0.922222,0.831481
3,삼성전자,기흥사업장,경기도 용인시 기흥구 삼성로 1(농서동),0.38430,37.226944,127.087764,1.065214,2.072041,2.263827,2.072041,0.484568,0.472222,1.045370,1.041667,1.087037
4,삼성전자,화성사업장,경기도 화성시 삼성전자로 1(반월동),0.91933,37.223465,127.063384,1.065214,2.072041,2.263827,2.072041,0.484568,0.472222,1.045370,1.041667,1.087037


['기업명', '사업장명', '사업장 위치', '면적(km²)', '위도', '경도', 'rfr_score', 'cfr_score', 'drr_score', 'B 점수', 'drr 백분위', 'B 백분위', 'drr 팩터', 'B 팩터', 'PRF']


## 2. 증가율 가정 정의

아래 증가율은 논문/문헌에서 제시된 장기 변화율을 연평균 증가율(CAGR)로 환산한 값을 사용한 예시

- `drr`: 가뭄위험. 유역별로 다르게 적용  
  - 금강권: 0.77%
  - 영산강/섬진강권: 0.27%
  - 기타 권역: 0.00%
- `rfr`: 하천범람위험. 기존 `rfr_score`의 수준별로 0.25%, 0.35%, 0.45%
- `cfr`: 해안침수위험. 기존 `cfr_score`의 수준별로 0.70%, 1.00%, 1.16%


```text
PRF = drr_factor + B_factor - 1
B_factor = max(rfr, cfr) 계열의 홍수위험
```


In [3]:
# 증가율 가정값
# 소수로 입력합니다. 예: 0.0077 = 0.77%
DRR_RATES = {
    "금강권": 0.0077,
    "영산강/섬진강권": 0.0027,
    "한강권/수도권": 0.0000,
    "낙동강권/동남권": 0.0000,
    "기타/확인필요": 0.0000,
}

# rfr_score 수준별 증가율
RFR_RATE_LOW = 0.0025   # 0.25%
RFR_RATE_MID = 0.0035   # 0.35%
RFR_RATE_HIGH = 0.0045  # 0.45%

# cfr_score 수준별 증가율
CFR_RATE_NONE = 0.0000  # cfr_score가 0이거나 결측
CFR_RATE_LOW = 0.0070   # 0.70%
CFR_RATE_MID = 0.0100   # 1.00%
CFR_RATE_HIGH = 0.0116  # 1.16%

DRR_RATES

{'금강권': 0.0077,
 '영산강/섬진강권': 0.0027,
 '한강권/수도권': 0.0,
 '낙동강권/동남권': 0.0,
 '기타/확인필요': 0.0}

## 3. 주소 기반 유역/권역 근사 매핑 함수

사업장 주소 문자열에 포함된 지역명을 기준으로 유역/권역을 근사 매핑

주의:
- `충북 청주`, `충남`, `대전`, `세종`, `전북 군산/익산/전주` 등은 금강권으로 근사
- `광주`, `전남`, `여수`, `광양`, `목포` 등은 영산강/섬진강권으로 근사
- 서울/경기/인천/강원은 한강권/수도권으로 근사
- 부산/울산/경남/대구/경북은 낙동강권/동남권으로 근사


In [4]:
def map_basin_from_address(address):
    """사업장 주소 문자열을 이용해 유역/권역을 근사 매핑합니다."""
    if pd.isna(address):
        return "기타/확인필요"

    addr = str(address).replace("\u00a0", " ").strip()

    # 금강권: 대전, 세종, 충남, 충북 청주/진천/음성 일부, 전북 북부 등
    geum_keywords = [
        "대전", "세종", "충청남도", "충남", "천안", "아산", "공주", "논산", "서산",
        "당진", "보령", "부여", "청양", "예산", "홍성",
        "청주", "진천", "음성", "옥천", "영동",
        "군산", "익산", "전주", "완주"
    ]

    # 영산강/섬진강권: 광주, 전남, 여수, 광양, 목포 등
    yeongsan_keywords = [
        "광주", "전라남도", "전남", "목포", "나주", "담양", "화순",
        "영암", "무안", "함평", "영광", "장성",
        "여수", "순천", "광양", "구례", "곡성", "보성"
    ]

    # 한강권/수도권: 서울, 경기, 인천, 강원
    han_keywords = [
        "서울", "경기도", "경기", "인천", "강원", "수원", "용인", "화성",
        "평택", "이천", "안성", "안산", "시흥", "파주", "남양주", "성남",
        "부천", "김포", "광명", "의왕", "군포", "오산"
    ]

    # 낙동강권/동남권: 부산, 울산, 경남, 경북, 대구
    nakdong_keywords = [
        "부산", "울산", "경상남도", "경남", "경상북도", "경북", "대구",
        "창원", "김해", "양산", "거제", "통영", "사천", "진주",
        "포항", "구미", "경주", "안동", "영주", "김천", "칠곡"
    ]

    if any(k in addr for k in geum_keywords):
        return "금강권"
    if any(k in addr for k in yeongsan_keywords):
        return "영산강/섬진강권"
    if any(k in addr for k in han_keywords):
        return "한강권/수도권"
    if any(k in addr for k in nakdong_keywords):
        return "낙동강권/동남권"

    return "기타/확인필요"


# 테스트
df[["사업장 위치"]].head(10).assign(유역=df["사업장 위치"].head(10).map(map_basin_from_address))

,사업장 위치,유역
0,경기도 수원시 영통구 삼성로 129(매탄동),한강권/수도권
1,서울특별시 서초구 서초대로74길 11(서초동),한강권/수도권
2,서울특별시 서초구 성촌길 33(우면동),한강권/수도권
3,경기도 용인시 기흥구 삼성로 1(농서동),한강권/수도권
4,경기도 화성시 삼성전자로 1(반월동),한강권/수도권
5,경기도 평택시 삼성로 114(고덕동),한강권/수도권
6,충청남도 천안시 서북구 번영로 465(성성동),금강권
7,충청남도 아산시 배방읍 배방로 158(북수리),금강권
8,충청남도 아산시 탕정면 삼성로 181(명암리),금강권
9,경상북도 구미시 1공단로 244(공단동),낙동강권/동남권


## 4. rfr/cfr 점수 기반 증가율 매핑 함수

`rfr_score`와 `cfr_score`는 사업장별 현재 위험수준을 나타내는 값입니다.  
여기서는 점수의 상대적 크기에 따라 낮음/중간/높음으로 나누어 증가율을 부여합니다.

- `rfr_score`: 전체 분포의 33%, 66% 분위수를 기준으로 3구간 분류
- `cfr_score`: 0은 해안침수 없음, 양수 값만 따로 분위수 기준으로 3구간 분류

이 방식은 점수의 현재 수준이 높을수록 미래 위험 증가율도 높게 반영하는 단순 규칙


In [5]:
def get_quantile_thresholds(series, exclude_zero=False):
    """점수 분포의 33%, 66% 분위수 기준값을 계산합니다."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if exclude_zero:
        s = s[s > 0]
    if len(s) == 0:
        return np.nan, np.nan
    return float(s.quantile(1/3)), float(s.quantile(2/3))


rfr_q33, rfr_q66 = get_quantile_thresholds(df["rfr_score"], exclude_zero=False)
cfr_q33, cfr_q66 = get_quantile_thresholds(df["cfr_score"], exclude_zero=True)

print("rfr 33%, 66% thresholds:", rfr_q33, rfr_q66)
print("cfr positive 33%, 66% thresholds:", cfr_q33, cfr_q66)


def map_rfr_rate(score):
    """rfr_score 수준별 하천범람 증가율을 부여합니다."""
    if pd.isna(score):
        return 0.0
    score = float(score)
    if score <= rfr_q33:
        return RFR_RATE_LOW
    elif score <= rfr_q66:
        return RFR_RATE_MID
    else:
        return RFR_RATE_HIGH


def map_cfr_rate(score):
    """cfr_score 수준별 해안침수 증가율을 부여합니다."""
    if pd.isna(score) or float(score) <= 0:
        return CFR_RATE_NONE
    score = float(score)
    if score <= cfr_q33:
        return CFR_RATE_LOW
    elif score <= cfr_q66:
        return CFR_RATE_MID
    else:
        return CFR_RATE_HIGH


def dominant_flood_type(row):
    """rfr_score와 cfr_score 중 더 큰 쪽을 지배 홍수위험으로 선택합니다."""
    rfr = row.get("rfr_score", np.nan)
    cfr = row.get("cfr_score", np.nan)
    rfr = -np.inf if pd.isna(rfr) else float(rfr)
    cfr = -np.inf if pd.isna(cfr) else float(cfr)
    return "CFR" if cfr > rfr else "RFR"


def map_b_rate(row):
    """지배 홍수위험이 RFR이면 rfr 증가율, CFR이면 cfr 증가율을 사용합니다."""
    if row["지배 홍수위험"] == "CFR":
        return row["cfr 증가율"]
    return row["rfr 증가율"]

rfr 33%, 66% thresholds: 1.06521394775 1.98927487286
cfr positive 33%, 66% thresholds: 2.07204141988 2.07204141988


## 5. 사업장별 세부 증가율 적용 및 미래 PRF 계산

기존 엑셀의 PRF 구조를 시간축으로 확장

```text
drr_factor(t) = drr_factor(0) × (1 + g_drr)^t
B_factor(t)   = B_factor(0) × (1 + g_B)^t

PRF(t) = drr_factor(t) + B_factor(t) - 1
```

기존처럼 `PRF(0)` 전체에 `1.02^t`를 곱하지 않고, `drr_factor`와 `B_factor`를 각각 다른 증가율로 성장시킨 뒤 다시 합산


In [6]:
# 원본 df를 복사해 작업
df_out = df.copy()

# 1) 주소 기반 유역/권역 매핑
df_out["유역/권역(근사)"] = df_out["사업장 위치"].map(map_basin_from_address)

# 2) 유역/권역에 따라 drr 증가율 부여
df_out["drr 증가율"] = df_out["유역/권역(근사)"].map(DRR_RATES).fillna(0.0)

# 3) rfr/cfr 점수 수준에 따라 증가율 부여
df_out["rfr 증가율"] = df_out["rfr_score"].map(map_rfr_rate)
df_out["cfr 증가율"] = df_out["cfr_score"].map(map_cfr_rate)

# 4) rfr/cfr 중 더 큰 쪽을 지배 홍수위험으로 선택
df_out["지배 홍수위험"] = df_out.apply(dominant_flood_type, axis=1)
df_out["B 증가율"] = df_out.apply(map_b_rate, axis=1)

# 5) 미래 PRF 계산
#    원본 열 이름: drr 팩터, B 팩터, PRF
for year in [10, 20, 30]:
    df_out[f"PRF_{year}Y"] = (
        pd.to_numeric(df_out["drr 팩터"], errors="coerce") * (1 + df_out["drr 증가율"]) ** year
        + pd.to_numeric(df_out["B 팩터"], errors="coerce") * (1 + df_out["B 증가율"]) ** year
        - 1
    )

# 6) 30년 기준 PRF 유효 CAGR
df_out["PRF 유효CAGR(30Y)"] = np.where(
    pd.to_numeric(df_out["PRF"], errors="coerce") > 0,
    (df_out["PRF_30Y"] / pd.to_numeric(df_out["PRF"], errors="coerce")) ** (1/30) - 1,
    np.nan
)

display(df_out.head())
df_out[["drr 증가율", "rfr 증가율", "cfr 증가율", "B 증가율", "PRF 유효CAGR(30Y)"]].describe()

,기업명,사업장명,사업장 위치,면적(km²),위도,경도,rfr_score,cfr_score,drr_score,B 점수,...,유역/권역(근사),drr 증가율,rfr 증가율,cfr 증가율,지배 홍수위험,B 증가율,PRF_10Y,PRF_20Y,PRF_30Y,PRF 유효CAGR(30Y)
0,삼성전자,수원사업장,경기도 수원시 영통구 삼성로 129(매탄동),1.30000,37.257643,127.053129,1.065214,2.072041,2.263827,2.072041,...,한강권/수도권,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734
1,삼성전자,서초사업장,서울특별시 서초구 서초대로74길 11(서초동),0.01340,37.496633,127.026914,0.562331,0.000000,1.965401,0.562331,...,한강권/수도권,0.0,0.0025,0.000,RFR,0.0025,0.854798,0.878704,0.903215,0.002762
2,삼성전자,우면사업장,서울특별시 서초구 성촌길 33(우면동),0.03720,37.465025,127.022762,0.562331,0.000000,1.965401,0.562331,...,한강권/수도권,0.0,0.0025,0.000,RFR,0.0025,0.854798,0.878704,0.903215,0.002762
3,삼성전자,기흥사업장,경기도 용인시 기흥구 삼성로 1(농서동),0.38430,37.226944,127.087764,1.065214,2.072041,2.263827,2.072041,...,한강권/수도권,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734
4,삼성전자,화성사업장,경기도 화성시 삼성전자로 1(반월동),0.91933,37.223465,127.063384,1.065214,2.072041,2.263827,2.072041,...,한강권/수도권,0.0,0.0025,0.007,CFR,0.0070,1.162294,1.242988,1.329512,0.006734


,drr 증가율,rfr 증가율,cfr 증가율,B 증가율,PRF 유효CAGR(30Y)
count,163.000000,163.000000,163.000000,163.000000,162.000000
mean,0.002471,0.003264,0.005172,0.005252,0.007279
std,0.003391,0.000910,0.004213,0.002645,0.003758
min,0.000000,0.000000,0.000000,0.000000,0.002762
25%,0.000000,0.002500,0.000000,0.003500,0.003964
50%,0.000000,0.002500,0.007000,0.004500,0.006734
75%,0.007700,0.004500,0.007000,0.007000,0.010089
max,0.007700,0.004500,0.011600,0.011600,0.016217


## 6. 매핑 근거/주의 컬럼 생성

각 사업장별로 어떤 규칙이 적용되었는지 간단한 설명 문자열을 붙임  
발표나 검토 과정에서 “왜 이 사업장이 이 증가율을 받았는지” 확인하는 데 사용


In [7]:
def rate_label(rate):
    """증가율을 % 문자열로 표시합니다."""
    if pd.isna(rate):
        return ""
    return f"{rate*100:.2f}%"


def make_note(row):
    basin = row["유역/권역(근사)"]
    drr_rate = rate_label(row["drr 증가율"])
    rfr_rate = rate_label(row["rfr 증가율"])
    cfr_rate = rate_label(row["cfr 증가율"])
    b_type = row["지배 홍수위험"]
    b_rate = rate_label(row["B 증가율"])

    return (
        f"주소 기반 수계 근사: {basin}, drr 증가율 {drr_rate}. "
        f"rfr 증가율 {rfr_rate}, cfr 증가율 {cfr_rate}. "
        f"지배 홍수위험은 {b_type}, B 증가율 {b_rate}. "
        "정밀 GIS 유역/해안/하천 거리 매핑은 후속 과제."
    )


df_out["매핑 근거/주의"] = df_out.apply(make_note, axis=1)

display(df_out[["기업명", "사업장명", "사업장 위치", "유역/권역(근사)", "drr 증가율", "rfr 증가율", "cfr 증가율", "지배 홍수위험", "B 증가율", "매핑 근거/주의"]].head(10))

,기업명,사업장명,사업장 위치,유역/권역(근사),drr 증가율,rfr 증가율,cfr 증가율,지배 홍수위험,B 증가율,매핑 근거/주의
0,삼성전자,수원사업장,경기도 수원시 영통구 삼성로 129(매탄동),한강권/수도권,0.0000,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
1,삼성전자,서초사업장,서울특별시 서초구 서초대로74길 11(서초동),한강권/수도권,0.0000,0.0025,0.000,RFR,0.0025,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
2,삼성전자,우면사업장,서울특별시 서초구 성촌길 33(우면동),한강권/수도권,0.0000,0.0025,0.000,RFR,0.0025,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
3,삼성전자,기흥사업장,경기도 용인시 기흥구 삼성로 1(농서동),한강권/수도권,0.0000,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
4,삼성전자,화성사업장,경기도 화성시 삼성전자로 1(반월동),한강권/수도권,0.0000,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
5,삼성전자,평택사업장,경기도 평택시 삼성로 114(고덕동),한강권/수도권,0.0000,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 한강권/수도권, drr 증가율 0.00%. rfr 증가율 0..."
6,삼성전자,천안사업장,충청남도 천안시 서북구 번영로 465(성성동),금강권,0.0077,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 금강권, drr 증가율 0.77%. rfr 증가율 0.25%..."
7,삼성전자,온양사업장,충청남도 아산시 배방읍 배방로 158(북수리),금강권,0.0077,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 금강권, drr 증가율 0.77%. rfr 증가율 0.25%..."
8,삼성전자,아산사업장,충청남도 아산시 탕정면 삼성로 181(명암리),금강권,0.0077,0.0025,0.007,CFR,0.0070,"주소 기반 수계 근사: 금강권, drr 증가율 0.77%. rfr 증가율 0.25%..."
9,삼성전자,구미1사업장,경상북도 구미시 1공단로 244(공단동),낙동강권/동남권,0.0000,0.0035,0.000,RFR,0.0035,"주소 기반 수계 근사: 낙동강권/동남권, drr 증가율 0.00%. rfr 증가율 ..."


## 7. 기업별 면적가중 PRF 요약 생성

사업장별 미래 PRF를 기업 단위로 요약
면적이 큰 사업장이 기업 전체 물리적 리스크에 더 크게 반영되도록 `면적(km²)` 기준 가중평균을 사용

```text
기업 PRF(t) = Σ[사업장 면적 × 사업장 PRF(t)] / Σ[사업장 면적]
```


In [8]:
def weighted_average(group, value_col, weight_col="면적(km²)"):
    """면적가중평균 계산. 면적이 없거나 0이면 단순평균으로 대체합니다."""
    values = pd.to_numeric(group[value_col], errors="coerce")
    weights = pd.to_numeric(group[weight_col], errors="coerce")
    mask = values.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0:
        return values.mean()
    return np.average(values[mask], weights=weights[mask])


company_rows = []
for company, g in df_out.groupby("기업명", dropna=False):
    total_area = pd.to_numeric(g["면적(km²)"], errors="coerce").sum()
    company_rows.append({
        "기업명": company,
        "사업장 수": len(g),
        "총 면적(km²)": total_area,
        "PRF_0Y": weighted_average(g, "PRF"),
        "PRF_10Y": weighted_average(g, "PRF_10Y"),
        "PRF_20Y": weighted_average(g, "PRF_20Y"),
        "PRF_30Y": weighted_average(g, "PRF_30Y"),
        "면적가중 drr 증가율": weighted_average(g, "drr 증가율"),
        "면적가중 rfr 증가율": weighted_average(g, "rfr 증가율"),
        "면적가중 cfr 증가율": weighted_average(g, "cfr 증가율"),
        "면적가중 B 증가율": weighted_average(g, "B 증가율"),
        "PRF 유효CAGR(30Y)": weighted_average(g, "PRF 유효CAGR(30Y)"),
        "지배 홍수위험": g["지배 홍수위험"].mode().iat[0] if not g["지배 홍수위험"].mode().empty else "",
    })

company_future = pd.DataFrame(company_rows)

display(company_future.head())
company_future.describe(include="all")

,기업명,사업장 수,총 면적(km²),PRF_0Y,PRF_10Y,PRF_20Y,PRF_30Y,면적가중 drr 증가율,면적가중 rfr 증가율,면적가중 cfr 증가율,면적가중 B 증가율,PRF 유효CAGR(30Y),지배 홍수위험
0,HD한국조선해양,2,2.7310,1.253022,1.328256,1.406512,1.487917,0.002076,0.004500,0.010537,0.004500,0.005765,RFR
1,HD현대미포,4,5.4490,1.337037,1.390698,1.446823,1.505526,0.000000,0.004500,0.007000,0.004500,0.003964,RFR
2,HD현대일렉트릭,3,0.2981,1.332341,1.386407,1.442994,1.502219,0.000000,0.004462,0.007000,0.004547,0.004016,RFR
3,HD현대중공업,1,4.7000,1.337037,1.390698,1.446823,1.505526,0.000000,0.004500,0.007000,0.004500,0.003964,RFR
4,HL만도,3,0.3740,1.007759,1.106919,1.215100,1.333217,0.002450,0.002818,0.005469,0.006539,0.008052,CFR


,기업명,사업장 수,총 면적(km²),PRF_0Y,PRF_10Y,PRF_20Y,PRF_30Y,면적가중 drr 증가율,면적가중 rfr 증가율,면적가중 cfr 증가율,면적가중 B 증가율,PRF 유효CAGR(30Y),지배 홍수위험
count,50,50.000000,50.000000,49.000000,49.000000,49.000000,49.000000,50.000000,50.000000,50.000000,50.000000,49.000000,50
unique,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,HD한국조선해양,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RFR
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33
mean,NaN,3.260000,5.591932,1.124074,1.209387,1.300949,1.399302,0.002142,0.003434,0.005895,0.005429,0.007253,NaN
std,NaN,2.926375,20.831426,0.144227,0.146983,0.161441,0.188640,0.002523,0.000901,0.003818,0.002638,0.003255,NaN
min,NaN,1.000000,0.000000,0.831481,0.854798,0.878704,0.903215,0.000000,0.000000,0.000000,0.000000,0.002762,NaN
25%,NaN,1.000000,0.710250,1.023035,1.112795,1.215100,1.329512,0.000000,0.002694,0.002793,0.003886,0.004013,NaN
50%,NaN,2.500000,1.000000,1.117645,1.211878,1.341573,1.479250,0.001390,0.003500,0.007000,0.004500,0.006572,NaN
75%,NaN,4.000000,2.697250,1.253022,1.339731,1.417479,1.505526,0.003149,0.004402,0.008316,0.006580,0.009448,NaN


## 8. 증가율 가정표 생성

엑셀에 별도 시트로 넣을 가정표

In [9]:
assumption_rows = [
    {
        "구분": "drr",
        "위험": "Drought Risk - 가뭄위험",
        "적용 기준": "금강권",
        "증가율": 0.0077,
        "설명": "가뭄위험 증가가 크게 나타나는 유역으로 별도 매핑",
        "주의": "주소 기반 근사. 정밀 유역 공간조인 필요",
    },
    {
        "구분": "drr",
        "위험": "Drought Risk - 가뭄위험",
        "적용 기준": "영산강/섬진강권",
        "증가율": 0.0027,
        "설명": "가뭄위험 증가가 관측되는 유역으로 별도 매핑",
        "주의": "주소 기반 근사. 정밀 유역 공간조인 필요",
    },
    {
        "구분": "drr",
        "위험": "Drought Risk - 가뭄위험",
        "적용 기준": "기타 권역",
        "증가율": 0.0000,
        "설명": "현 단계에서는 문헌 기반 증가율을 적용하지 않음",
        "주의": "전국 평균값이 아니라 유역별 근사값",
    },
    {
        "구분": "rfr",
        "위험": "Riverine Flood Risk - 하천범람위험",
        "적용 기준": "rfr_score 낮음",
        "증가율": RFR_RATE_LOW,
        "설명": "100년 빈도 홍수 증가율을 연평균으로 환산한 보수 구간",
        "주의": "점수 분포 분위수 기반",
    },
    {
        "구분": "rfr",
        "위험": "Riverine Flood Risk - 하천범람위험",
        "적용 기준": "rfr_score 중간",
        "증가율": RFR_RATE_MID,
        "설명": "기준 구간",
        "주의": "점수 분포 분위수 기반",
    },
    {
        "구분": "rfr",
        "위험": "Riverine Flood Risk - 하천범람위험",
        "적용 기준": "rfr_score 높음",
        "증가율": RFR_RATE_HIGH,
        "설명": "고위험 구간",
        "주의": "점수 분포 분위수 기반",
    },
    {
        "구분": "cfr",
        "위험": "Coastal Flood Risk - 해안침수위험",
        "적용 기준": "cfr_score = 0",
        "증가율": CFR_RATE_NONE,
        "설명": "해안침수 노출이 없는 것으로 처리",
        "주의": "cfr_score 품질에 의존",
    },
    {
        "구분": "cfr",
        "위험": "Coastal Flood Risk - 해안침수위험",
        "적용 기준": "cfr_score 낮음",
        "증가율": CFR_RATE_LOW,
        "설명": "전체 침수면적 증가율 기준에 가까운 보수값",
        "주의": "점수 분포 분위수 기반",
    },
    {
        "구분": "cfr",
        "위험": "Coastal Flood Risk - 해안침수위험",
        "적용 기준": "cfr_score 중간",
        "증가율": CFR_RATE_MID,
        "설명": "개발지역 피해면적 증가율 기준값",
        "주의": "점수 분포 분위수 기반",
    },
    {
        "구분": "cfr",
        "위험": "Coastal Flood Risk - 해안침수위험",
        "적용 기준": "cfr_score 높음",
        "증가율": CFR_RATE_HIGH,
        "설명": "도로/인프라 피해 증가율 기준에 가까운 고위험값",
        "주의": "점수 분포 분위수 기반",
    },
]

assumptions = pd.DataFrame(assumption_rows)
display(assumptions)

,구분,위험,적용 기준,증가율,설명,주의
0,drr,Drought Risk - 가뭄위험,금강권,0.0077,가뭄위험 증가가 크게 나타나는 유역으로 별도 매핑,주소 기반 근사. 정밀 유역 공간조인 필요
1,drr,Drought Risk - 가뭄위험,영산강/섬진강권,0.0027,가뭄위험 증가가 관측되는 유역으로 별도 매핑,주소 기반 근사. 정밀 유역 공간조인 필요
2,drr,Drought Risk - 가뭄위험,기타 권역,0.0000,현 단계에서는 문헌 기반 증가율을 적용하지 않음,전국 평균값이 아니라 유역별 근사값
3,rfr,Riverine Flood Risk - 하천범람위험,rfr_score 낮음,0.0025,100년 빈도 홍수 증가율을 연평균으로 환산한 보수 구간,점수 분포 분위수 기반
4,rfr,Riverine Flood Risk - 하천범람위험,rfr_score 중간,0.0035,기준 구간,점수 분포 분위수 기반
5,rfr,Riverine Flood Risk - 하천범람위험,rfr_score 높음,0.0045,고위험 구간,점수 분포 분위수 기반
6,cfr,Coastal Flood Risk - 해안침수위험,cfr_score = 0,0.0000,해안침수 노출이 없는 것으로 처리,cfr_score 품질에 의존
7,cfr,Coastal Flood Risk - 해안침수위험,cfr_score 낮음,0.0070,전체 침수면적 증가율 기준에 가까운 보수값,점수 분포 분위수 기반
8,cfr,Coastal Flood Risk - 해안침수위험,cfr_score 중간,0.0100,개발지역 피해면적 증가율 기준값,점수 분포 분위수 기반
9,cfr,Coastal Flood Risk - 해안침수위험,cfr_score 높음,0.0116,도로/인프라 피해 증가율 기준에 가까운 고위험값,점수 분포 분위수 기반


## 9. 엑셀 파일로 저장

원본 workbook의 기존 시트는 유지하고, 아래 내용을 반영

- `사업장별 PRF`: 기존 15개 열 뒤에 신규 매핑/미래 PRF 컬럼 추가
- `PRF_증가율_가정`: 증가율 가정과 매핑 기준 정리
- `기업별 PRF_미래`: 기업별 면적가중 미래 PRF 요약


In [10]:
# 원본 workbook 불러오기
wb = load_workbook(input_path)

# 기존 '사업장별 PRF' 시트 교체를 위해 삭제 후 재생성
if site_sheet in wb.sheetnames:
    del wb[site_sheet]
ws = wb.create_sheet(site_sheet, 1)

# DataFrame을 worksheet에 쓰는 helper
def write_dataframe(ws, data):
    # headers
    for c_idx, col in enumerate(data.columns, start=1):
        ws.cell(row=1, column=c_idx, value=col)
    # values
    for r_idx, row in enumerate(data.itertuples(index=False), start=2):
        for c_idx, value in enumerate(row, start=1):
            if pd.isna(value):
                value = None
            ws.cell(row=r_idx, column=c_idx, value=value)

write_dataframe(ws, df_out)

# 가정표 시트 생성/교체
assumption_sheet = "PRF_증가율_가정"
if assumption_sheet in wb.sheetnames:
    del wb[assumption_sheet]
ws_assump = wb.create_sheet(assumption_sheet)
write_dataframe(ws_assump, assumptions)

# 기업별 요약 시트 생성/교체
company_sheet = "기업별 PRF_미래"
if company_sheet in wb.sheetnames:
    del wb[company_sheet]
ws_company = wb.create_sheet(company_sheet)
write_dataframe(ws_company, company_future)

# 간단한 스타일 적용
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin_gray = Side(style="thin", color="D9E2F3")
border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

for sheet in [ws, ws_assump, ws_company]:
    # 헤더 스타일
    for cell in sheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border

    # 틀고정 및 필터
    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions

    # 전체 셀 테두리/정렬
    for row in sheet.iter_rows():
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="center", wrap_text=True)

    # 열 너비 조정
    for col_idx, column_cells in enumerate(sheet.columns, start=1):
        max_len = 0
        for cell in column_cells:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))
        width = min(max(max_len + 2, 10), 45)
        sheet.column_dimensions[get_column_letter(col_idx)].width = width

# 숫자 포맷
percent_cols_site = ["drr 증가율", "rfr 증가율", "cfr 증가율", "B 증가율", "PRF 유효CAGR(30Y)"]
for col_name in percent_cols_site:
    col_idx = df_out.columns.get_loc(col_name) + 1
    for row in range(2, ws.max_row + 1):
        ws.cell(row=row, column=col_idx).number_format = "0.00%"

for col_name in ["면적가중 drr 증가율", "면적가중 rfr 증가율", "면적가중 cfr 증가율", "면적가중 B 증가율", "PRF 유효CAGR(30Y)"]:
    if col_name in company_future.columns:
        col_idx = company_future.columns.get_loc(col_name) + 1
        for row in range(2, ws_company.max_row + 1):
            ws_company.cell(row=row, column=col_idx).number_format = "0.00%"

if "증가율" in assumptions.columns:
    col_idx = assumptions.columns.get_loc("증가율") + 1
    for row in range(2, ws_assump.max_row + 1):
        ws_assump.cell(row=row, column=col_idx).number_format = "0.00%"

# 저장
wb.save(output_path)
print(f"Saved: {output_path}")

Saved: /Users/dapanman/Downloads/물리적리스크_PRF_세부증가율_매핑.xlsx
